<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/Copy_of_building_semantic_search_handson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Semantic Search — Hands-on Lab

### From Dense → Sparse → Hybrid → Reranking with LangChain

**Runtime:** CPU (GPU recommended for faster reranking)

| Section | Activity |
| :--- | :--- |
| 0 | **Setup + Toy Corpus** |
| 1 | **Dense Retrieval Failures** |
| 2 | **Sparse Retrieval (BM25)** |
| 3 | **Hybrid Search (RRF)** |
| 4 | **Cross-Encoder Reranking** |
| 5 | **Evaluation Metrics (Recall, MRR, MAP)** |

> **Key Takeaway:** Implementing a **hybrid + rerank** pipeline usually yields better results than simply upgrading to a more expensive embedding model.

---

## 0. Setup

No API keys or external billing are required.

*   **langchain / langchain-community:** Core retrieval interfaces and ensemble utilities.
*   **sentence-transformers:** Provides the **bi-encoder** (embeddings) and **cross-encoder** (reranker).
*   **rank_bm25:** Powering the sparse keyword search.
*   **faiss-cpu:** Our in-memory vector database.


In [ ]:
# Quiet install — the -q flags keep the output readable during a live session.
!pip install -q langchain langchain-community langchain-huggingface \
               sentence-transformers rank_bm25 faiss-cpu

print("Done. If Colab shows a 'restart runtime' banner, you can safely ignore it here.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Done. If Colab shows a 'restart runtime' banner, you can safely ignore it here.


In [ ]:
# Re-verifying and installing specific sub-packages to resolve import issues
!pip install -U -q langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 9.8 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain-classic

### The Toy Corpus

We use twelve support-desk documents designed to demonstrate specific search challenges:

1.  **The Identifier Trap (`ERR_2043`):** Embedding models often struggle with unique codes, blurring `ERR_2043` and `ERR_2049` together.
2.  **The Synonym Trap ("paid leave" vs "vacation"):** Keyword search fails when different words share the same meaning.

Each document includes `metadata` (`dept`, `year`) for our filtering exercises in Section 6.

In [ ]:
from langchain_core.documents import Document

# Each Document = page_content (the text that gets indexed) + metadata (used for filtering).
raw_docs = [
    # --- the exact-identifier trap -------------------------------------------------
    (
        "ERR_2043: database connection timeout. Increase the connection pool size in "
        "config/db.yaml and restart the service.",
        {"dept": "ENG", "year": 2025}
    ),
    (
        "ERR_2049: authentication token expired. Re-issue the token via the admin console.",
        {"dept": "ENG", "year": 2025}
    ),
    (
        "General guidance on error handling and best practices: log the stack trace, fail "
        "loudly, and surface actionable messages to the user.",
        {"dept": "ENG", "year": 2024}
    ),
    # --- the synonym trap ----------------------------------------------------------
    (
        "Employees accrue paid leave at 1.5 days per month. Unused days roll over once, "
        "up to a cap of 30 days.",
        {"dept": "HR", "year": 2025}
    ),
    (
        "To request time away from work, file the form in the HR portal at least two weeks "
        "in advance.",
        {"dept": "HR", "year": 2024}
    ),
    # --- distractors and filler ----------------------------------------------------
    (
        "The expense reimbursement policy covers travel, client meals, and conference "
        "tickets. Submit receipts within 30 days.",
        {"dept": "FIN", "year": 2025}
    ),
    (
        "Parental leave is 26 weeks fully paid for the primary caregiver.",
        {"dept": "HR", "year": 2025}
    ),
    (
        "calc_tax_v2() replaces the deprecated calc_tax() helper. It takes a jurisdiction "
        "code and returns a Decimal.",
        {"dept": "ENG", "year": 2025}
    ),
    (
        "Quarterly revenue targets are set by the finance committee each January.",
        {"dept": "FIN", "year": 2024}
    ),
    (
        "Reset your workstation password from the self-service portal; passwords expire "
        "every 90 days.",
        {"dept": "IT", "year": 2025}
    ),
    (
        "The VPN client must be updated to version 4.2 before the end of the quarter.",
        {"dept": "IT", "year": 2025}
    ),
    (
        "Database backups run nightly at 02:00 UTC and are retained for 35 days.",
        {"dept": "ENG", "year": 2024}
    ),
]

docs = [Document(page_content=text, metadata=meta) for text, meta in raw_docs]

print(f"{len(docs)} documents indexed.\n")
for i, d in enumerate(docs[:3]):
    print(f"[{i}] ({d.metadata['dept']}) {d.page_content[:70]}...")

12 documents indexed.

[0] (ENG) ERR_2043: database connection timeout. Increase the connection pool si...
[1] (ENG) ERR_2049: authentication token expired. Re-issue the token via the adm...
[2] (ENG) General guidance on error handling and best practices: log the stack t...


### Display Helper

This utility ensures that results across all retrieval methods are printed in a consistent, readable format for easy comparison.

In [ ]:
from langchain_core.documents import Document

def show(results: list[Document], title: str = "", width: int = 88) -> None:
    """Pretty-print a list of Documents as a ranked list.

    Args:
        results: A list of Document objects to display.
        title: An optional title to print before the results.
        width: The maximum width for the snippet of each document's page_content.
    """
    if title:
        print(f"\n{title}\n" + "-" * len(title))
    for rank, doc in enumerate(results, start=1):
        snippet = doc.page_content[:width].replace("\n", " ")
        dept = doc.metadata.get("dept", "?")
        print(f"{rank:>2}. [{dept:<3}] {snippet}...")

---

## 1. Dense Retrieval

**Concept:** A **bi-encoder** embeds queries and documents into a shared vector space independently. Retrieval is performed using Approximate nearest-neighbor(ANN) search. This method is highly scalable as document vectors are precomputed.

We are using the `all-MiniLM-L6-v2` model: a small, efficient, and capable transformer for semantic search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# The bi-encoder. Downloads ~90MB the first time.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# FAISS builds an in-memory vector index over our 12 documents.
vectorstore = FAISS.from_documents(docs, embeddings)

# .as_retriever() wraps the store in LangChain's standard Retriever interface,
# so every retriever in this notebook is called the same way: .invoke(query)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k":3})

print("Dense index ready.")

/tmp/ipykernel_1258/2416361790.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dense index ready.


### 1a. Semantic Matching

**Query:** *"how much vacation do I get?"*

The word **"vacation"** does not appear in our documents. However, dense search should successfully find the **"paid leave"** policy by understanding the underlying meaning.

In [ ]:
show(
    dense_retriever.invoke("how much vacation do I get?"),
    title="DENSE  ·  'how much vacation do I get?'"
)


DENSE  ·  'how much vacation do I get?'
---------------------------------------
 1. [HR ] Employees accrue paid leave at 1.5 days per month. Unused days roll over once, up to a c...
 2. [HR ] Parental leave is 26 weeks fully paid for the primary caregiver....
 3. [FIN] The expense reimbursement policy covers travel, client meals, and conference tickets. Su...


✅ **Success:** It found "paid leave" despite zero keyword overlap.

### 1b. The Blind Spot: Exact Identifiers

**Query:** *"How do I fix error ERR_2043?"*

Because `ERR_2043` lacks semantic meaning to the model, it often confuses it with `ERR_2049` or generic error handling docs.

In [ ]:
show(
    dense_retriever.invoke("How do I fix error ERR_2043?"),
    title="DENSE  ·  'How do I fix error ERR_2043?'"
)


DENSE  ·  'How do I fix error ERR_2043?'
----------------------------------------
 1. [ENG] ERR_2049: authentication token expired. Re-issue the token via the admin console....
 2. [ENG] ERR_2043: database connection timeout. Increase the connection pool size in config/db.ya...
 3. [ENG] General guidance on error handling and best practices: log the stack trace, fail loudly,...


❌ **Failure:** Exact identifiers often get buried under semantic noise. This is where **Sparse Search** excels.

---

## 2. Sparse Retrieval (BM25)

**Concept:** BM25 scores documents based on literal word matches. It prioritizes terms that are frequent in a specific document but rare across the entire collection (**TF-IDF** logic).

In [ ]:
from langchain_community.retrievers import BM25Retriever

# k = how many documents to return. BM25Retriever tokenises on whitespace by default.
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

print("Sparse Retriever is Ready!")

show(
    sparse_retriever.invoke("ERR_2043"),
    title="SPARSE (BM25)  ·  'ERR_2043'"
)

Sparse Retriever is Ready!

SPARSE (BM25)  ·  'ERR_2043'
----------------------------
 1. [ENG] Database backups run nightly at 02:00 UTC and are retained for 35 days....
 2. [IT ] The VPN client must be updated to version 4.2 before the end of the quarter....
 3. [IT ] Reset your workstation password from the self-service portal; passwords expire every 90 ...


✅ **Success:** `ERR_2043` is rare, so the specific document containing it is easily retrieved.

### 2a. The Mirror Failure

If we search for **"vacation"** using BM25, it will fail because the literal word is missing.

In [ ]:
show(
    sparse_retriever.invoke("how much vacation do I get?"),
    title="SPARSE (BM25)  ·  'how much vacation do I get?'"
)


SPARSE (BM25)  ·  'how much vacation do I get?'
-----------------------------------------------
 1. [ENG] Database backups run nightly at 02:00 UTC and are retained for 35 days....
 2. [IT ] The VPN client must be updated to version 4.2 before the end of the quarter....
 3. [IT ] Reset your workstation password from the self-service portal; passwords expire every 90 ...


❌ **Failure:** Sparse search has no concept of synonyms.

**Summary of Blind Spots:**

| Scenario | Dense | Sparse |
| :--- | :---: | :---: |
| **Synonyms / Paraphrasing** | ✅ | ❌ |
| **Exact Rare Tokens (SKUs, IDs)** | ❌ | ✅ |

---

## 3. Hybrid Search

By running both retrievers in parallel and fusing their results, we eliminate the blind spots of both methods.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

results = hybrid_retriever.invoke("How do I fix error ERR_2043?")

for doc in results:
    print(doc.page_content)
    print(doc.metadata)

General guidance on error handling and best practices: log the stack trace, fail loudly, and surface actionable messages to the user.
{'dept': 'ENG', 'year': 2024}
ERR_2049: authentication token expired. Re-issue the token via the admin console.
{'dept': 'ENG', 'year': 2025}
ERR_2043: database connection timeout. Increase the connection pool size in config/db.yaml and restart the service.
{'dept': 'ENG', 'year': 2025}
Database backups run nightly at 02:00 UTC and are retained for 35 days.
{'dept': 'ENG', 'year': 2024}
Reset your workstation password from the self-service portal; passwords expire every 90 days.
{'dept': 'IT', 'year': 2025}


In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

# The same two queries that each broke one of the retrievers:
show(
    hybrid_retriever.invoke("ERR_2043?"),
    title="HYBRID  ·  'ERR_2043?'"
)

show(
    hybrid_retriever.invoke("how much vacation do I get?"),
    title="HYBRID  ·  'how much vacation do I get?'"
)


HYBRID  ·  'ERR_2043?'
----------------------
 1. [ENG] ERR_2049: authentication token expired. Re-issue the token via the admin console....
 2. [ENG] Database backups run nightly at 02:00 UTC and are retained for 35 days....
 3. [ENG] ERR_2043: database connection timeout. Increase the connection pool size in config/db.ya...
 4. [IT ] The VPN client must be updated to version 4.2 before the end of the quarter....
 5. [ENG] calc_tax_v2() replaces the deprecated calc_tax() helper. It takes a jurisdiction code an...
 6. [IT ] Reset your workstation password from the self-service portal; passwords expire every 90 ...

HYBRID  ·  'how much vacation do I get?'
----------------------------------------
 1. [HR ] Employees accrue paid leave at 1.5 days per month. Unused days roll over once, up to a c...
 2. [ENG] Database backups run nightly at 02:00 UTC and are retained for 35 days....
 3. [HR ] Parental leave is 26 weeks fully paid for the primary caregiver....
 4. [IT ] The VPN client must

✅ **Outcome:** Both types of queries now work correctly. Use **Hybrid Search** whenever your queries mix natural language with technical identifiers (e.g., SKUs, error codes, legal terms).

### Explanation: Reciprocal Rank Fusion (RRF)

**Reciprocal Rank Fusion (RRF)** is an algorithm used to combine the results of multiple search engines (or retrievers) into a single unified list without needing to normalize their underlying scores (like cosine similarity vs. BM25 scores).

#### How it works:
1.  **Inverse Ranking:** For every document retrieved by one of your search methods, RRF calculates a score based on its rank: $Score = \frac{1}{k + Rank}$.
2.  **Damping Factor ($k$):** The constant $k$ (usually set to 60) prevents high-ranked documents from completely dominating the results and reduces the impact of noise from low-ranked documents.
3.  **Summation:** If a document appears in both the **Dense** and **Sparse** results, its scores are added together. Documents that appear near the top of *multiple* lists will naturally rise to the very top of the final fused list.

In your code:
*   `scores[key] += 1.0 / (k + rank)` implements the formula.
*   It allows you to benefit from both semantic understanding (Dense) and keyword matching (Sparse) simultaneously.

In [ ]:
from collections import defaultdict
from langchain_core.documents import Document

def reciprocal_rank_fusion(
    ranked_lists: list[list[Document]], k: int = 60
) -> list[tuple[Document, float]]:
    """Fuses multiple ranked lists of documents using Reciprocal Rank Fusion (RRF).

    Args:
        ranked_lists: A list of ranked Document lists, where each inner list
                      comes from a different retriever.
        k: A damping constant; 60 is the value from the original RRF paper.
           Larger k = flatter weighting across ranks.

    Returns:
        A list of tuples, where each tuple contains a Document and its fused score,
        sorted by fused score in descending order.
    """
    scores = defaultdict(float)
    lookup = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):  # rank is 1-based
            key = doc.page_content  # dedupe key across lists
            scores[key] += 1.0 / (k + rank)
            lookup[key] = doc

    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [(lookup[key], score) for key, score in ordered]

query = "ERR_2043"
fused = reciprocal_rank_fusion(
    [
        dense_retriever.invoke(query),
        sparse_retriever.invoke(query),
    ]
)
print(f"RRF fusion  ·  '{query}'\n" + "-" * 60)
for rank, (doc, score) in enumerate(fused, start=1):
    print(f"{rank:>2}. score={score:.4f}  {doc.page_content[:60]}...")

RRF fusion  ·  'ERR_2043'
------------------------------------------------------------
 1. score=0.0164  ERR_2049: authentication token expired. Re-issue the token v...
 2. score=0.0164  Database backups run nightly at 02:00 UTC and are retained f...
 3. score=0.0161  ERR_2043: database connection timeout. Increase the connecti...
 4. score=0.0161  The VPN client must be updated to version 4.2 before the end...
 5. score=0.0159  General guidance on error handling and best practices: log t...
 6. score=0.0159  Reset your workstation password from the self-service portal...


---
## 4. Reranking with a cross-encoder
**The two-stage pattern:** retrieve fast and approximately (top ~100 from millions), then re-scorethat shortlist slowly and precisely.The difference from Section
1:
- **Bi-encoder** (retrieval): encodes query and document *separately* → precomputable → scales.
- **Cross-encoder** (reranking): feeds query and document *together* through one model, so it sees  word-by-word interaction → far more accurate, but must run once **per pair** at query time.  Too slow for a million docs; perfectly affordable for 100.In LangChain, a reranker wraps a base retriever via `ContextualCompressionRetriever`.

## Resources

*   [LangChain Retrieval Documentation](https://python.langchain.com/docs/modules/data_connection/retrievers/)
*   [Sentence Transformers (Hugging Face)](https://huggingface.co/sentence-transformers)
*   [FAISS Documentation](https://faiss.ai/)

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import EnsembleRetriever

# cross-encoder
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

compressor = CrossEncoderReranker(model=cross_encoder, top_n=3)

wide_dense = vectorstore.as_retriever(search_kwargs={"k":8})
wide_sparse = BM25Retriever.from_documents(docs)
wide_sparse.k = 8

wide_hybrid = EnsembleRetriever(
    retrievers=[wide_dense, wide_sparse],
    weights=[0.5, 0.5]
)

reranked_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=wide_hybrid
)

print("Reranker ready")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Reranker ready


In [ ]:
query = "my database keeps timing out, what do I change?"

# Note this query has NO error code in it - it's a natural-language description.
show(
    wide_hybrid.invoke(query)[:8],
    title=f"BEFORE rerank (hybrid top-8)  ·  '{query}'"
)

show(
    reranked_retriever.invoke(query),
    title=f"AFTER rerank (cross-encoder top-3)  ·  '{query}'"
)


BEFORE rerank (hybrid top-8)  ·  'my database keeps timing out, what do I change?'
----------------------------------------------------------------------------------
 1. [ENG] ERR_2043: database connection timeout. Increase the connection pool size in config/db.ya...
 2. [ENG] Database backups run nightly at 02:00 UTC and are retained for 35 days....
 3. [IT ] Reset your workstation password from the self-service portal; passwords expire every 90 ...
 4. [IT ] The VPN client must be updated to version 4.2 before the end of the quarter....
 5. [HR ] To request time away from work, file the form in the HR portal at least two weeks in adv...
 6. [ENG] ERR_2049: authentication token expired. Re-issue the token via the admin console....
 7. [FIN] Quarterly revenue targets are set by the finance committee each January....
 8. [ENG] General guidance on error handling and best practices: log the stack trace, fail loudly,...

AFTER rerank (cross-encoder top-3)  ·  'my database keeps timing out

The cross-encoder pushes the connection-pool fix up, because it can actually read "timing out"against "connection timeout ... increase the pool size" as a *pair*, rather than comparing twoindependently-made vectors.**The cost:** stage 1 is one embedding lookup; stage 2 is N forward passes through a transformer.That is why you only ever rerank a shortlist.> *Also worth knowing (not run here):* **ColBERT** sits between these two — one vector per **token**> rather than per document, scored by summing each query token's best match (MaxSim). More precise> than a single doc vector, far cheaper than a cross-encoder, but storage-hungry.

---

## 5. Measuring Retrieval Quality


*   **Recall@K:** Did the correct document appear in the top K results? (Critical for RAG).
*   **Precision@K:** How many of the top K results were actually relevant?
*   **MRR (Mean Reciprocal Rank):** How high was the *first* relevant result ranked?
*   **MAP (Mean Average Precision):** Rewards systems that rank multiple relevant documents highly.

In [ ]:
def precision_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    """Calculates Precision@K.

    Args:
        retrieved: A list of identifiers for retrieved documents, ordered by rank.
        relevant: A set of identifiers for relevant documents.
        k: The number of top documents to consider.

    Returns:
        The precision at K.
    """
    top_k = retrieved[:k]
    return sum(1 for d in top_k if d in relevant) / k if k else 0.0


def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    """Calculates Recall@K.

    Args:
        retrieved: A list of identifiers for retrieved documents, ordered by rank.
        relevant: A set of identifiers for relevant documents.
        k: The number of top documents to consider.

    Returns:
        The recall at K.
    """
    if not relevant:
        return 0.0
    top_k = retrieved[:k]
    return sum(1 for d in top_k if d in relevant) / len(relevant)


def reciprocal_rank(retrieved: list[str], relevant: set[str]) -> float:
    """Calculates Reciprocal Rank (RR).

    Args:
        retrieved: A list of identifiers for retrieved documents, ordered by rank.
        relevant: A set of identifiers for relevant documents.

    Returns:
        1 / (rank of the first relevant result); 0 if none found.
    """
    for rank, d in enumerate(retrieved, start=1):
        if d in relevant:
            return 1.0 / rank
    return 0.0


def average_precision(retrieved: list[str], relevant: set[str]) -> float:
    """Calculates Average Precision (AP).

    Args:
        retrieved: A list of identifiers for retrieved documents, ordered by rank.
        relevant: A set of identifiers for relevant documents.

    Returns:
        The average precision.
    """
    if not relevant:
        return 0.0
    hits, total = 0, 0.0
    for rank, d in enumerate(retrieved, start=1):
        if d in relevant:
            hits += 1
            total += hits / rank  # precision at this position
    return total / len(relevant)


# Sanity check against the worked example from the slides:
# relevant hits at ranks 1, 3, 6 out of 3 relevant docs total.
demo = ["A", "x", "B", "x", "x", "C"]
gold = {"A", "B", "C"}
print(f"Precision@3 = {precision_at_k(demo, gold, 3):.2f}   (expected 0.67)")
print(f"Recall@6    = {recall_at_k(demo, gold, 6):.2f}   (expected 1.00)")
print(f"MRR         = {reciprocal_rank(demo, gold):.2f}   (expected 1.00)")
print(f"AveP        = {average_precision(demo, gold):.2f}   (expected 0.72)")

Precision@3 = 0.67   (expected 0.67)
Recall@6    = 1.00   (expected 1.00)
MRR         = 1.00   (expected 1.00)
AveP        = 0.72   (expected 0.72)
